# Étude Technico-Économique : Dimensionnement Optimal du Champ Solaire
## Microgrid Portuaire — Analyse de Sensibilité de la Puissance Installée

---

### Objectif

Ce notebook présente une **étude paramétrique** visant à identifier la puissance nominale optimale du champ photovoltaïque (PV) dans le contexte du microgrid portuaire. La variable de décision est la capacité installée du champ solaire, balayée de **0 à 40 MWp** par pas de **0,5 MW**.

Pour chaque configuration, un réseau PyPSA est construit, optimisé (dispatch optimal), puis son bilan technico-économique est calculé. L'optimum est défini comme la configuration **maximisant le bilan net annuel** de la municipalité.

### Hypothèses de modélisation

- La batterie est **incluse** dans toutes les simulations (`Battery=True`), conformément à la topologie de référence.
- Tous les autres paramètres (capacité de stockage, tarif Shore Power, fiscalité) sont fixés aux valeurs nominales du fichier `config.py`.
- Le coût annualisé du solaire est calculé via la formule CRF (Capital Recovery Factor) avec un taux d'actualisation de 6 % sur 25 ans.
- Le bilan financier intègre : dépenses énergétiques, CAPEX+OPEX solaire et batterie, revenus de revente Shore Power.

---

> **Note technique :** La fonction `create_pypsa_network` expose un paramètre `PV_nominal_capacity` mais lit en interne `cfg.SOLAR_CAPACITY_MW`. La stratégie adoptée ici consiste à mettre à jour dynamiquement cette variable de configuration avant chaque instanciation du réseau, garantissant la cohérence du modèle.

---
## 0. Imports & Configuration de l'environnement

In [1]:
# ── Bibliothèques standard ──────────────────────────────────────────────────
import warnings
import logging

# ── Calcul scientifique ────────────────────────────────────────────────────
import numpy as np
import pandas as pd

# ── Visualisation ──────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.lines import Line2D

# ── Barre de progression ───────────────────────────────────────────────────
from tqdm.notebook import tqdm

# ── Modules projet ─────────────────────────────────────────────────────────
import src.config as cfg
import src._1_data_prep as dp
import src._2_power_flow_optimization as pfo
from src._3_overall_cost import generer_bilan_financier

# ── Silencer les logs PyPSA/linopy (réduire le bruit en console) ───────────
warnings.filterwarnings('ignore')
logging.getLogger('pypsa').setLevel(logging.ERROR)
logging.getLogger('linopy').setLevel(logging.ERROR)

# ── Style graphique ────────────────────────────────────────────────────────
plt.style.use('seaborn-v0_8-whitegrid')
COLORS = {
    'solar'   : '#F4A261',  # Orange solaire
    'bilan'   : '#2A9D8F',  # Vert bilan
    'depense' : '#E76F51',  # Rouge dépenses
    'revenu'  : '#457B9D',  # Bleu revenus
    'optimal' : '#E9C46A',  # Jaune optimum
    'neutre'  : '#6C757D',  # Gris neutre
}

print("✅ Environnement initialisé.")

✅ Environnement initialisé.


---
## 1. Chargement et préparation des données

In [2]:
# Chargement du jeu de données complet
df=dp.sort_ship_columns(dp.load_and_prepare_data(),6,0,69)


# ── Contrôle qualité ───────────────────────────────────────────────────────
print(f"Période simulée       : {df.index[0].date()} → {df.index[-1].date()}")
print(f"Nombre de snapshots   : {len(df):,} ({len(df)/8760:.2f} années)")
print(f"Colonnes de base      : {list(df.columns[:6])}")
print(f"Nombre de navires     : {len(df.columns) - 6}")
print(f"\nStatistiques des charges (MWh) :\n")
display(df.describe().round(3))

Période simulée       : 2024-01-01 → 2024-12-31
Nombre de snapshots   : 8,784 (1.00 années)
Colonnes de base      : ['private_MWh', 'price_EUR_MWh', 'business_MWh', 'CO2_g_MWh', 't2m_C', 'radiation_solaire_factor']
Nombre de navires     : 69

Statistiques des charges (MWh) :



,private_MWh,price_EUR_MWh,business_MWh,CO2_g_MWh,t2m_C,radiation_solaire_factor,AIDANOVA,MSC EURIBIA,IONA,NORWEGIAN PRIMA,...,EUROPA,SEABOURN VENTURE,STAR PRIDE,DEUTSCHLAND,HAMBURG,CORINTHIAN,WORLD VOYAGER,HANSEATIC SPIRIT,HEBRIDEAN SKY,NOORDERLICHT
count,8784.000,8784.000,8784.000,8784.000,8784.000,8784.000,8784.000,8784.000,8784.000,8784.000,...,8784.000,8784.000,8784.000,8784.000,8784.000,8784.000,8784.000,8784.000,8784.000,8784.000
mean,0.153,20.091,0.990,6722.487,7.526,0.096,0.247,0.238,0.172,0.169,...,0.002,0.002,0.001,0.001,0.001,0.001,0.001,0.001,0.000,0.000
std,0.053,19.055,0.281,2245.361,5.328,0.160,1.931,1.886,1.616,1.415,...,0.063,0.054,0.040,0.045,0.036,0.017,0.027,0.031,0.010,0.001
min,0.056,5.000,0.488,10.000,-8.129,0.000,0.000,0.000,0.000,0.000,...,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
25%,0.106,11.000,0.776,5310.000,3.600,0.000,0.000,0.000,0.000,0.000,...,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
50%,0.151,15.000,0.948,6830.000,7.657,0.003,0.000,0.000,0.000,0.000,...,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
75%,0.193,21.950,1.158,8250.000,12.069,0.132,0.000,0.000,0.000,0.000,...,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
max,0.309,285.900,1.949,14660.000,22.584,0.760,15.366,15.169,15.382,11.993,...,2.414,1.922,1.086,1.880,1.259,0.341,0.830,1.298,0.351,0.012


---
## 2. Définition de l'espace paramétrique

L'espace de recherche est défini sur $[0, 40]$ MWp avec un pas de résolution de $\Delta P = 0{,}5$ MW, soit **81 points de simulation**.

In [3]:
# ── Espace de variation de la puissance nominale PV ────────────────────────
PV_MIN_MW   = 0.0    # Borne inférieure [MWp]
PV_MAX_MW   = 40.0   # Borne supérieure [MWp]
PV_STEP_MW  = 0.5    # Pas de variation  [MWp]

pv_range = np.arange(PV_MIN_MW, PV_MAX_MW + PV_STEP_MW, PV_STEP_MW)

print(f"Espace paramétrique   : [{PV_MIN_MW}, {PV_MAX_MW}] MWp")
print(f"Pas de discrétisation : {PV_STEP_MW} MW")
print(f"Nombre de simulations : {len(pv_range)}")

Espace paramétrique   : [0.0, 40.0] MWp
Pas de discrétisation : 0.5 MW
Nombre de simulations : 81


---
## 3. Boucle de simulation paramétrique

Pour chaque valeur de puissance nominale $P_{PV}$ :
1. La variable `cfg.SOLAR_CAPACITY_MW` est mise à jour dynamiquement.
2. Un réseau PyPSA est instancié avec `Battery=True`.
3. L'optimisation linéaire du dispatch est résolue via `network.optimize()`.
4. Le bilan financier annuel est extrait et stocké.

In [ ]:
# ── Initialisation du conteneur de résultats ───────────────────────────────
results = []

# ── Boucle paramétrique ────────────────────────────────────────────────────
for pv_capacity in tqdm(pv_range, desc="Simulation PV sizing", unit="config"):

    # ── (a) Mise à jour dynamique de la configuration ──────────────────────
    # La fonction create_pypsa_network lit cfg.SOLAR_CAPACITY_MW en interne.
    # On synchronise ici la variable globale avec le paramètre courant.
    cfg.SOLAR_CAPACITY_MW = pv_capacity

    # ── (b) Construction du réseau PyPSA ──────────────────────────────────
    network = pfo.create_pypsa_network(
        df,
        PV_nominal_capacity=pv_capacity,
        Battery=True,
        index_ship=6
    )

    # ── (c) Optimisation du dispatch (programmation linéaire) ──────────────
    # solver_name peut être adapté : 'highs' (défaut OSS), 'glpk', 'gurobi'
    status, condition = network.optimize(solver_name='highs')

    # ── (d) Vérification du statut d'optimisation ──────────────────────────
    if status != 'ok':
        print(f"⚠️  Optimisation non convergée pour PV = {pv_capacity} MW (status: {status}, condition: {condition})")
        results.append({
            'pv_capacity_MW'    : pv_capacity,
            'status'            : status,
            'depenses_energie'  : np.nan,
            'depenses_solaire'  : np.nan,
            'depenses_stockage' : np.nan,
            'total_depenses'    : np.nan,
            'revenu_sp'         : np.nan,
            'bilan'             : np.nan,
        })
        continue

    # ── (e) Calcul du bilan financier annuel ──────────────────────────────
    bilan_dict = generer_bilan_financier(network)

    # ── (f) Stockage des résultats ─────────────────────────────────────────
    results.append({
        'pv_capacity_MW'    : pv_capacity,
        'status'            : status,
        **bilan_dict          # Déplie toutes les clés du bilan
    })

# ── Conversion en DataFrame d'analyse ─────────────────────────────────────
df_results = pd.DataFrame(results).set_index('pv_capacity_MW')

print(f"\n✅ Simulations terminées. {df_results['status'].value_counts().get('ok', 0)} cas convergés.")

Simulation PV sizing:   0%|          | 0/81 [00:00<?, ?config/s]

Writing continuous variables.: 100%|██████████| 5/5 [00:00<00:00, 737.58it/s]


Running HiGHS 1.13.1 (git hash: 1d267d9): Copyright (c) 2026 under MIT licence terms
LP linopy-problem-wm9hzqr5 has 219600 rows; 87840 cols; 333791 nonzeros
Coefficient ranges:
  Matrix  [9e-01, 1e+00]
  Cost    [1e-02, 2e+02]
  Bound   [0e+00, 0e+00]
  RHS     [1e-02, 1e+06]
Presolving model
26352 rows, 52704 cols, 96623 nonzeros  0s
17568 rows, 35136 cols, 61487 nonzeros  0s
Dependent equations search running on 16287 equations with time limit of 1000.00s
Dependent equations search removed 0 rows and 0 nonzeros in 0.00s (limit = 1000.00s)
16287 rows, 33855 cols, 57644 nonzeros  0s
Presolve reductions: rows 16287(-203313); columns 33855(-53985); nonzeros 57644(-276147) 
Solving the presolved LP
Using dual simplex solver
  Iteration        Objective     Infeasibilities num(sum)
          0     0.0000000000e+00 Ph1: 0(0) 0.1s
      25368     4.6891086006e+06 Pr: 0(0); Du: 0(6.76244e-13) 0.3s

Performed postsolve
Solving the original LP from the solution after postsolve

Model name      

Writing continuous variables.: 100%|██████████| 5/5 [00:00<00:00, 733.45it/s]


Running HiGHS 1.13.1 (git hash: 1d267d9): Copyright (c) 2026 under MIT licence terms
LP linopy-problem-3hnprdl5 has 219600 rows; 87840 cols; 333791 nonzeros
Coefficient ranges:
  Matrix  [9e-01, 1e+00]
  Cost    [1e-02, 2e+02]
  Bound   [0e+00, 0e+00]
  RHS     [9e-06, 1e+06]
Presolving model
31123 rows, 67017 cols, 120478 nonzeros  0s
27110 rows, 54220 cols, 94884 nonzeros  0s
Dependent equations search running on 22283 equations with time limit of 1000.00s
Dependent equations search removed 0 rows and 0 nonzeros in 0.00s (limit = 1000.00s)
22283 rows, 49393 cols, 85174 nonzeros  0s
Presolve reductions: rows 22283(-197317); columns 49393(-38447); nonzeros 85174(-248617) 
Solving the presolved LP
Using dual simplex solver
  Iteration        Objective     Infeasibilities num(sum)
          0     0.0000000000e+00 Ph1: 0(0) 0.1s
      33266     4.6279093864e+06 Pr: 0(0); Du: 0(1.03915e-12) 0.4s

Performed postsolve
Solving the original LP from the solution after postsolve

Model name     

...

Writing continuous variables.: 100%|██████████| 5/5 [00:00<00:00, 734.97it/s]


Running HiGHS 1.13.1 (git hash: 1d267d9): Copyright (c) 2026 under MIT licence terms
LP linopy-problem-mciu2rm8 has 219600 rows; 87840 cols; 333791 nonzeros
Coefficient ranges:
  Matrix  [9e-01, 1e+00]
  Cost    [1e-02, 2e+02]
  Bound   [0e+00, 0e+00]
  RHS     [3e-04, 1e+06]
Presolving model
31123 rows, 67017 cols, 120478 nonzeros  0s
27110 rows, 54220 cols, 94884 nonzeros  0s
Dependent equations search running on 22283 equations with time limit of 1000.00s
Dependent equations search removed 0 rows and 0 nonzeros in 0.00s (limit = 1000.00s)
22283 rows, 49393 cols, 85174 nonzeros  0s
Presolve reductions: rows 22283(-197317); columns 49393(-38447); nonzeros 85174(-248617) 
Solving the presolved LP
Using dual simplex solver
  Iteration        Objective     Infeasibilities num(sum)
          0     0.0000000000e+00 Ph1: 0(0) 0.1s
      32145     3.2565426815e+06 Pr: 0(0); Du: 0(1.63102e-13) 0.4s

Performed postsolve
Solving the original LP from the solution after postsolve

Model name     

Writing continuous variables.: 100%|██████████| 5/5 [00:00<00:00, 749.30it/s]


Running HiGHS 1.13.1 (git hash: 1d267d9): Copyright (c) 2026 under MIT licence terms
LP linopy-problem-uffcl8tc has 219600 rows; 87840 cols; 333791 nonzeros
Coefficient ranges:
  Matrix  [9e-01, 1e+00]
  Cost    [1e-02, 2e+02]
  Bound   [0e+00, 0e+00]
  RHS     [3e-04, 1e+06]
Presolving model
31123 rows, 67017 cols, 120478 nonzeros  0s
27110 rows, 54220 cols, 94884 nonzeros  0s
Dependent equations search running on 22283 equations with time limit of 1000.00s
Dependent equations search removed 0 rows and 0 nonzeros in 0.00s (limit = 1000.00s)
22283 rows, 49393 cols, 85174 nonzeros  0s
Presolve reductions: rows 22283(-197317); columns 49393(-38447); nonzeros 85174(-248617) 
Solving the presolved LP
Using dual simplex solver
  Iteration        Objective     Infeasibilities num(sum)
          0     0.0000000000e+00 Ph1: 0(0) 0.1s
      32091     3.2252108969e+06 Pr: 0(0); Du: 0(2.04671e-13) 0.4s

Performed postsolve
Solving the original LP from the solution after postsolve

Model name     

Writing continuous variables.: 100%|██████████| 5/5 [00:00<00:00, 752.45it/s]

---
## 4. Analyse des résultats

### 4.1 Tableau récapitulatif

In [ ]:
# ── Tableau de synthèse (valeurs en k€) ───────────────────────────────────
df_display = (df_results
              .drop(columns=['status'], errors='ignore')
              .div(1000)                    # Conversion € → k€
              .round(1))

df_display.columns = [
    'Dép. Énergie [k€]',
    'Dép. Solaire [k€]',
    'Dép. Stockage [k€]',
    'Total Dépenses [k€]',
    'Revenu SP [k€]',
    'Bilan Net [k€]'
]

# Affichage avec mise en forme conditionnelle du bilan
display(
    df_display.style
    .background_gradient(subset=['Bilan Net [k€]'], cmap='RdYlGn')
    .format("{:,.1f}")
    .set_caption("Tableau 1 — Résultats financiers annuels par configuration PV")
)

### 4.2 Identification de l'optimum

In [ ]:
# ── Filtrage des cas convergés ─────────────────────────────────────────────
df_valid = df_results[df_results['status'] == 'ok'].copy()

# ── Recherche de l'optimum global ─────────────────────────────────────────
idx_opt   = df_valid['bilan'].idxmax()
bilan_opt = df_valid.loc[idx_opt, 'bilan']

# ── Recherche du seuil de rentabilité (bilan > 0) ─────────────────────────
df_positif  = df_valid[df_valid['bilan'] >= 0]
pv_min_rent = df_positif.index.min() if not df_positif.empty else None
pv_max_rent = df_positif.index.max() if not df_positif.empty else None

# ── Configuration de référence (valeur nominale du config.py) ──────────────
PV_REF_MW   = 10.0   # Valeur originale de SOLAR_CAPACITY_MW
bilan_ref   = df_valid.loc[PV_REF_MW, 'bilan'] if PV_REF_MW in df_valid.index else np.nan

# ── Affichage synthétique ──────────────────────────────────────────────────
print("━" * 55)
print("         SYNTHÈSE TECHNICO-ÉCONOMIQUE")
print("━" * 55)
print(f"  Configuration optimale    : {idx_opt:>6.1f} MWp")
print(f"  Bilan net maximal         : {bilan_opt/1e3:>+10.1f} k€/an")
print("─" * 55)
if pv_min_rent is not None:
    print(f"  Plage de rentabilité       : [{pv_min_rent:.1f} – {pv_max_rent:.1f}] MWp")
else:
    print("  Plage de rentabilité       : aucune configuration profitable")
print("─" * 55)
print(f"  Référence (config.py)      : {PV_REF_MW:>6.1f} MWp")
print(f"  Bilan de référence         : {bilan_ref/1e3:>+10.1f} k€/an")
print(f"  Gain vs référence          : {(bilan_opt - bilan_ref)/1e3:>+10.1f} k€/an")
print("━" * 55)

---
## 5. Visualisations

### Figure 1 — Bilan net annuel en fonction de la puissance PV installée

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))

bilan_k = df_valid['bilan'] / 1e3

# ── Courbe principale ──────────────────────────────────────────────────────
ax.plot(df_valid.index, bilan_k,
        color=COLORS['bilan'], linewidth=2.2, zorder=3, label='Bilan net annuel')

# ── Zone profitable (bilan > 0) ────────────────────────────────────────────
ax.fill_between(df_valid.index, bilan_k, 0,
                where=(bilan_k >= 0),
                alpha=0.18, color=COLORS['bilan'], label='Zone profitable')
ax.fill_between(df_valid.index, bilan_k, 0,
                where=(bilan_k < 0),
                alpha=0.18, color=COLORS['depense'], label='Zone déficitaire')

# ── Point optimal ─────────────────────────────────────────────────────────
ax.scatter(idx_opt, bilan_opt / 1e3,
           s=120, color=COLORS['optimal'], edgecolors='k', zorder=5,
           label=f'Optimum : {idx_opt:.1f} MWp ({bilan_opt/1e3:+.0f} k€/an)')

# ── Configuration de référence ─────────────────────────────────────────────
if not np.isnan(bilan_ref):
    ax.axvline(PV_REF_MW, color=COLORS['neutre'], linestyle='--',
               linewidth=1.4, label=f'Référence config.py ({PV_REF_MW} MWp)')

# ── Ligne de neutralité ────────────────────────────────────────────────────
ax.axhline(0, color='black', linewidth=0.9, linestyle='-')

# ── Mise en forme des axes ─────────────────────────────────────────────────
ax.set_xlabel('Puissance nominale du champ PV [MWp]', fontsize=12)
ax.set_ylabel('Bilan net annuel [k€/an]', fontsize=12)
ax.set_title('Figure 1 — Sensibilité du bilan financier à la capacité solaire installée\n'
             '(Batterie incluse, tarif Shore Power fixe)',
             fontsize=13, fontweight='bold', pad=12)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
ax.set_xlim(PV_MIN_MW, PV_MAX_MW)
ax.legend(fontsize=10, loc='best', framealpha=0.85)
ax.tick_params(labelsize=10)

plt.tight_layout()
plt.savefig('figure1_bilan_vs_pv.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure 1 sauvegardée → figure1_bilan_vs_pv.png")

### Figure 2 — Décomposition des flux financiers (revenus, dépenses, bilan)

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(11, 9), sharex=True)

# ── (a) Décomposition des dépenses en aires empilées ──────────────────────
ax1 = axes[0]
dep_energie  = df_valid['depenses_energie']  / 1e3
dep_solaire  = df_valid['depenses_solaire']  / 1e3
dep_stockage = df_valid['depenses_stockage'] / 1e3
revenu_sp    = df_valid['revenu_sp']          / 1e3

ax1.stackplot(df_valid.index,
              dep_energie, dep_solaire, dep_stockage,
              labels=['Dép. énergétique (réseau)', 'CAPEX+OPEX Solaire', 'CAPEX+OPEX Batterie'],
              colors=['#E76F51', '#F4A261', '#2A9D8F'],
              alpha=0.82)
ax1.plot(df_valid.index, revenu_sp,
         color=COLORS['revenu'], linewidth=2.2, linestyle='-',
         label='Revenu Shore Power (constant)')

ax1.set_ylabel('Montant annuel [k€/an]', fontsize=11)
ax1.set_title('(a) Décomposition des dépenses et revenus annuels', fontsize=11, fontweight='bold')
ax1.legend(fontsize=9, loc='upper right', framealpha=0.85)
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
ax1.tick_params(labelsize=10)

# ── (b) Bilan net et ses composantes ──────────────────────────────────────
ax2 = axes[1]
bilan_k       = df_valid['bilan']         / 1e3
total_dep_k   = df_valid['total_depenses'] / 1e3

ax2.plot(df_valid.index, bilan_k,
         color=COLORS['bilan'], linewidth=2.4, label='Bilan net')
ax2.plot(df_valid.index, -dep_solaire,
         color='#F4A261', linewidth=1.6, linestyle='--',
         label='Impact négatif CAPEX+OPEX Solaire')
ax2.fill_between(df_valid.index, bilan_k, 0,
                 where=(bilan_k >= 0), alpha=0.15, color=COLORS['bilan'])
ax2.fill_between(df_valid.index, bilan_k, 0,
                 where=(bilan_k < 0),  alpha=0.15, color=COLORS['depense'])

# Point optimal
ax2.scatter(idx_opt, bilan_opt / 1e3,
            s=120, color=COLORS['optimal'], edgecolors='k', zorder=5,
            label=f'Optimum : {idx_opt:.1f} MWp')
ax2.axhline(0, color='black', linewidth=0.9)

ax2.set_xlabel('Puissance nominale du champ PV [MWp]', fontsize=11)
ax2.set_ylabel('Bilan net annuel [k€/an]', fontsize=11)
ax2.set_title('(b) Évolution du bilan net en fonction de la capacité PV', fontsize=11, fontweight='bold')
ax2.legend(fontsize=9, loc='best', framealpha=0.85)
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
ax2.set_xlim(PV_MIN_MW, PV_MAX_MW)
ax2.tick_params(labelsize=10)

fig.suptitle('Figure 2 — Décomposition technico-économique du microgrid portuaire\n'
             'en fonction de la puissance PV installée',
             fontsize=13, fontweight='bold', y=1.01)

plt.tight_layout()
plt.savefig('figure2_decomposition_financiere.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure 2 sauvegardée → figure2_decomposition_financiere.png")

### Figure 3 — Analyse marginale : rendement marginal de chaque MW supplémentaire

In [ ]:
# ── Calcul du bilan marginal (dérivée discrète) ────────────────────────────
# Représente le gain (ou la perte) de bilan net apporté par 1 MW supplémentaire
bilan_marginal = df_valid['bilan'].diff() / PV_STEP_MW / 1e3  # k€ / MW

fig, ax = plt.subplots(figsize=(11, 4.5))

ax.bar(bilan_marginal.index, bilan_marginal,
       width=PV_STEP_MW * 0.85,
       color=np.where(bilan_marginal >= 0, COLORS['bilan'], COLORS['depense']),
       alpha=0.85, label='Δ Bilan / MW supplémentaire')

ax.axhline(0, color='black', linewidth=0.9)

# Annotation de l'optimum (Δbilan = 0)
ax.axvline(idx_opt, color=COLORS['optimal'], linestyle='--', linewidth=1.6,
           label=f'Optimum ({idx_opt:.1f} MWp)')

ax.set_xlabel('Puissance nominale du champ PV [MWp]', fontsize=12)
ax.set_ylabel('Bilan marginal [k€ / MW]', fontsize=12)
ax.set_title('Figure 3 — Rendement marginal du champ solaire\n'
             '(gain de bilan net par MW supplémentaire installé)',
             fontsize=13, fontweight='bold', pad=10)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.1f}'))
ax.set_xlim(PV_MIN_MW, PV_MAX_MW)
ax.legend(fontsize=10, framealpha=0.85)
ax.tick_params(labelsize=10)

plt.tight_layout()
plt.savefig('figure3_bilan_marginal.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure 3 sauvegardée → figure3_bilan_marginal.png")

---
## 6. Export des résultats

In [ ]:
# ── Export CSV complet ─────────────────────────────────────────────────────
output_csv = 'resultats_dimensionnement_pv.csv'
df_results.to_csv(output_csv, sep=';', decimal=',', encoding='utf-8-sig')
print(f"Résultats exportés → {output_csv}")

# ── Récapitulatif des fichiers produits ────────────────────────────────────
print("\nFichiers générés :")
print(f"  📊  {output_csv}")
print(f"  🖼️   figure1_bilan_vs_pv.png")
print(f"  🖼️   figure2_decomposition_financiere.png")
print(f"  🖼️   figure3_bilan_marginal.png")

---
## 7. Conclusions et recommandations

Cette analyse paramétrique permet de dégager les enseignements suivants :

1. **Configuration optimale** : la puissance installée maximisant le bilan net annuel est identifiée automatiquement (cellule §4.2). Au-delà de ce point, l'augmentation du CAPEX+OPEX solaire n'est plus compensée par les économies sur la facture réseau, conduisant à une dégradation du bilan.

2. **Rendement marginal décroissant** (Figure 3) : l'apport de chaque mégawatt supplémentaire diminue à mesure que le champ solaire couvre une fraction croissante de la demande locale. Le changement de signe de la dérivée discrète localise précisément l'optimum.

3. **Seuil de rentabilité** : la plage $[P_{min}^{rent}, P_{max}^{rent}]$ définit les configurations pour lesquelles le projet est financièrement positif pour la municipalité. Cette information est clé pour la prise de décision.

4. **Sensibilité** : les résultats dépendent des hypothèses de coût (CAPEX solaire, tarif Shore Power, fiscalité). Une analyse de robustesse multi-scénarios (optimiste / central / pessimiste) est recommandée en prolongement.

### Perspectives

- Coupler ce balayage PV avec une étude similaire sur la capacité de batterie (surface 2D d'optimisation).
- Intégrer des contraintes supplémentaires : taux d'autoconsommation minimal, limitation d'injection réseau, ou objectifs carbone.
- Mener une analyse de sensibilité sur le tarif `SHORE_POWER_COST_PER_MW` et le paramètre fiscal `TAX_PRICE`.